# Make an audio summary of a scientific article

In [14]:
from pathlib import Path

# Resolve paths relative to the project root (one level up from this notebook)
PROJECT_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent

# Input
PAPER_PDF = PROJECT_ROOT / "papers/UNIFORM_2025.pdf"
SKILL_FILE = PROJECT_ROOT / ".github/skills/paper-to-audio-summary/SKILL.md"

# Output
OUTPUT_DIR = PROJECT_ROOT / "output"

In [ ]:
# Import libraries
import os
import numpy as np
from pypdf import PdfReader
from kokoro import KPipeline
from dotenv import load_dotenv
import litellm

load_dotenv()  # Load environment variables from .env file

True

In [16]:
def check_environment_variables():
    api_keys = ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"]
    missing_vars = [var for var in api_keys if var not in os.environ]
    if len(missing_vars) == len(api_keys):
        raise EnvironmentError(f"Must have one of the required environment variables: {', '.join(api_keys)}")

### Step 1: Convert PDF to Text

In [4]:
def convert_pdf_to_text(pdf_path: str, output_file: str=None) -> str:
    """
    Convert a PDF file to text and save it to the specified output file.
    Returns the extracted text.
    """
    # Load the PDF file
    reader = PdfReader(pdf_path)
    
    # Loop and extract text from all pages
    text = ""
    for _, page in enumerate(reader.pages):
        text += page.extract_text()

    if output_file:
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "w") as f:
            f.write(text)

    return text


paper_txt_file = os.path.join(OUTPUT_DIR, str(Path(PAPER_PDF).stem) + ".txt")
paper_text = convert_pdf_to_text(PAPER_PDF, paper_txt_file)
print(f"Saved {paper_txt_file}")

Saved /Users/ang/GitHub/yt-paper-to-audio/output/UNIFORM_2025.txt


### Step 2: Prepare the user prompt for the LLM

In [8]:
def make_user_input(paper_txt: str, skill_file: str) -> str:
    # Read the skill file
    with open(skill_file, "r") as f:
        skill_content = f.read()

    # Create user input by combining the paper text and the skill content
    user_input = f"Skill Instructions:\n{skill_content}\n\nPaper Text: {paper_txt}\n\nSummary:"
    return user_input

user_input = make_user_input(paper_text, SKILL_FILE)

### Step 3: Generate Paper Summary

In [9]:
# Configure LLM using litellm (provider-agnostic).
# Set MODEL in your .env, e.g.:
#   MODEL=anthropic/claude-sonnet-4-6   (requires ANTHROPIC_API_KEY)
#   MODEL=openai/gpt-4o-mini            (requires OPENAI_API_KEY)
model = os.environ.get("MODEL", "anthropic/claude-sonnet-4-6")
print(f"Using model: {model}")


def get_llm_response(
    prompt: str,
    model: str = model,
    temperature: float = 0.0,
    max_tokens: int = 5000,
) -> str:
    response = litellm.completion(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()


response_text = get_llm_response(user_input)

# Save
response_file = os.path.join(OUTPUT_DIR, str(Path(PAPER_PDF).stem) + "_summary.txt")
with open(response_file, "w") as f:
    f.write(response_text)
print(f"Saved {response_file}")

Using model: anthropic/claude-sonnet-4-6
Saved /Users/ang/GitHub/yt-paper-to-audio/output/UNIFORM_2025_summary.txt


### Step 4: Convert Text to Audio

In [27]:
from tqdm.auto import tqdm
import re


def convert_text_to_speech(text: str, output_file: str = None) -> np.ndarray:
    """
    Convert text to speech using Kokoro and save it to the specified output file.
    """
    pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')

    # Split into sentences so tqdm can show per-sentence progress
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    audio_chunks = []
    for sentence in tqdm(sentences, desc="Synthesizing audio", unit="sentence"):
        for _, _, audio in pipeline(sentence, voice='af_heart', speed=1.0):
            audio_chunks.append(audio)

    full_audio = np.concatenate(audio_chunks)

    if output_file:
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, "wb") as f:
            f.write(full_audio.tobytes())
        print(f"Saved {output_file}")

    return full_audio

**Test audio conversion**

In [28]:
from IPython.display import Audio

# Process a small test sample
test_text = "Hello! This is a test of the Kokoro text to speech model."
test_audio = convert_text_to_speech(test_text)

# Preview the generated audio in the notebook
Audio(test_audio, rate=24000)

Synthesizing audio: 100%|██████████| 2/2 [00:00<00:00,  2.37sentence/s]


**Convert full paper summary**

In [29]:
# Create and save audio for the paper summary
summary_audio_file = os.path.join(OUTPUT_DIR, str(Path(PAPER_PDF).stem) + "_summary.wav")
full_audio = convert_text_to_speech(response_text, summary_audio_file)

Synthesizing audio: 100%|██████████| 131/131 [03:00<00:00,  1.38s/sentence]

Saved /Users/ang/GitHub/yt-paper-to-audio/output/UNIFORM_2025_summary.wav


### Convert to MP3 format

This is better for portability- results in a smaller file size

In [31]:
import soundfile as sf
import io
from pydub import AudioSegment

def audio_to_mp3(audio_data, sample_rate=24000):
    wav_buffer = io.BytesIO()
    sf.write(wav_buffer, audio_data, sample_rate, format='WAV')
    wav_buffer.seek(0)
    segment = AudioSegment.from_wav(wav_buffer)
    mp3_buffer = io.BytesIO()
    segment.export(mp3_buffer, format='mp3', bitrate='192k')
    mp3_buffer.seek(0)
    return mp3_buffer.getvalue()

mp3_bytes = audio_to_mp3(full_audio)

# Save the MP3 audio file
summary_audio_file_mp3 = os.path.join(OUTPUT_DIR, str(Path(PAPER_PDF).stem) + "_summary.mp3")
with open(summary_audio_file_mp3, 'wb') as f:
    f.write(mp3_bytes)
print(f'Saved {summary_audio_file_mp3}')

Saved /Users/ang/GitHub/yt-paper-to-audio/output/UNIFORM_2025_summary.mp3
